# A Thousand Observations Are Not a Thousand Replicates

Synthetic experiments on dependent samples, effective sample size, HAC uncertainty, moving-block bootstrap and forward forecast validation. See the article and README for statistical assumptions.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
from analysis import (Scenario, stationary_ar1, exact_mean_variance,
                      effective_sample_size, asymptotic_long_run_variance,
                      iid_mean_se, hac_mean_se, normal_mean_interval,
                      moving_block_means, basic_bootstrap_interval,
                      one_forecast_validation, run_study, default_bandwidth)
scenario = Scenario()
print('n:', scenario.n, 'phi:', scenario.main_phi)

In [ ]:
# Exact finite-sample precision-equivalent iid sample size.
phi, n = scenario.main_phi, scenario.n
print('Exact variance inflation:', n * exact_mean_variance(phi, n))
print('Effective iid sample count:', effective_sample_size(phi, n))
print('Asymptotic long-run variance:', asymptotic_long_run_variance(phi))

In [ ]:
# A single dependent path: naive IID, HAC and moving-block intervals.
series = stationary_ar1(np.random.default_rng(42), n, phi)[0]
lag = default_bandwidth(n)
iid_ci = normal_mean_interval(series, iid_mean_se(series))
hac_ci = normal_mean_interval(series, hac_mean_se(series, lag))
blocks = moving_block_means(np.random.default_rng(43), series, lag, 800)
mbb_ci = basic_bootstrap_interval(series, blocks)
print('IID:', np.round(iid_ci, 3), 'HAC:', np.round(hac_ci, 3), 'MBB:', np.round(mbb_ci, 3))

In [ ]:
# Random mixed-regime validation and genuine future validation differ.
forecast, signal = one_forecast_validation(np.random.default_rng(49))
for name, value in forecast.items():
    print(name, round(value, 3))

In [ ]:
# Reproduce small results in a temporary directory, without overwriting tracked evidence.
with TemporaryDirectory() as folder:
    location = Path(folder)
    stats = run_study(quick=True, output_dir=location)
    assert len(list((location / 'figures').glob('*.png'))) == 8
    assert len(list((location / 'results').glob('*.csv'))) == 10
    assert stats['naive_coverage_phi_0_8'] < stats['hac_coverage_phi_0_8']
    print('Quick-study IID coverage:', stats['naive_coverage_phi_0_8'])

## Important scope

The stationary confidence-interval formulas assume a common mean, short-range dependence and appropriate bandwidth or block length. The forecast experiment is intentionally nonstationary; its random mixed-regime holdout answers a different question from future performance. These simulated numbers do not establish real deployment accuracy.